In [1]:
import random
import os
from google.colab import drive
import json
import sys
drive.mount('/content/drive')
%cd content
%cd drive
%cd MyDrive
if os.path.exists("Standards") == False:
  os.mkdir("Standards")
if os.path.exists("SyntheticGenes") == False:
  os.mkdir("SyntheticGenes")

%run GeneClassesCloud.ipynb


Mounted at /content/drive
[Errno 2] No such file or directory: 'content'
/content
/content/drive
/content/drive/MyDrive


In [2]:
def generate_codon_vec(aaseq: str):
  '''Take in an aa seq and output a vector of codons that capture the combination space'''
  aaCodonVecs = {
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
    'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'C': ['TGT', 'TGC'],
    'W': ['TGG'],
    'E': ['GAA', 'GAG'],
    'D': ['GAT', 'GAC'],
    'P': ['CCT', 'CCC', 'CCA', 'CCG'],
    'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'N': ['AAT', 'AAC'],
    'M': ['ATG'],
    'K': ['AAA', 'AAG'],
    'Y': ['TAT', 'TAC'],
    'I': ['ATT', 'ATC', 'ATA'],
    'Q': ['CAA', 'CAG'],
    'F': ['TTT', 'TTC'],
    'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    '*': ['TAA', 'TAG', 'TGA'],
    'A': ['GCT', 'GCC', 'GCA', 'GCG'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG'],
    'H': ['CAT', 'CAC']
  }
  codonvec=[]
  for aa in aaseq.upper():
    if aa not in aaCodonVecs.keys():
      raise NotImplementedError
    else:
      codonvec.append(aaCodonVecs[aa])
  return codonvec

def standards_check():
  '''Check for analysis class objects in the standards directory'''

  #Check to see if Standards/RareCodonAnalysis.json exists. If so, load it
  if os.path.exists(os.path.join("Standards","RareCodonAnalysis.json")):
    with open(os.path.join("Standards","RareCodonAnalysis.json"), "r") as f:
      dta = json.load(f)
    rare_codon_analysis = RareCodonAnalysis(**dta)
  #If not, run the script then load the results
  else:
    %run Rare_Codons.ipynb
    with open(os.path.join("Standards","RareCodonAnalysis.json"), "r") as f:
      dta = json.load(f)
    rare_codon_analysis = RareCodonAnalysis(**dta)


  #Check to see if Standards/CodonUsageAnalysis.json exists. If so, load it
  if os.path.exists(os.path.join("Standards","CodonUsageAnalysis.json")):
    with open(os.path.join("Standards","CodonUsageAnalysis.json"), "r") as f:
      dta = json.load(f)
    codon_analysis = CodonAnalysis(**dta)
  #If not, run the script then load the results
  else:
    %run Codon_Usage.ipynb
    with open(os.path.join("Standards","CodonUsageAnalysis.json"), "r") as f:
      dta = json.load(f)
    codon_analysis = CodonAnalysis(**dta)


  #Check to see if Standards/CodonUsageAnalysis.json exists. If so, load it
  if os.path.exists(os.path.join("Standards","CodonPairBiasAnalysis.json")):
    with open(os.path.join("Standards","CodonPairBiasAnalysis.json"), "r") as f:
      dta = json.load(f)
    cpb_analysis = CodonPairBiasAnalysis(**dta)
  #If not, run the script then load the results
  else:
    %run Codon_Pair_Bias.ipynb
    with open(os.path.join("Standards","CodonPairBiasAnalysis.json"), "r") as f:
      dta = json.load(f)
    cpb_analysis = CodonPairBiasAnalysis(**dta)


  #Check to see if Standards/CodonUsageAnalysis.json exists. If so, load it
  if os.path.exists(os.path.join("Standards","GCAnalysis.json")):
    with open(os.path.join("Standards","GCAnalysis.json"), "r") as f:
      dta = json.load(f)
    gc_analysis = GCAnalysis(**dta)
  #If not, run the script then load the results
  else:
    %run GC_Analysis.ipynb
    with open(os.path.join("Standards","GCAnalysis.json"), "r") as f:
      dta = json.load(f)
    gc_analysis = GCAnalysis(**dta)


  #Check to see if Standards/CodonUsageAnalysis.json exists. If so, load it
  if os.path.exists(os.path.join("Standards","KmerAnalysis.json")):
    with open(os.path.join("Standards","KmerAnalysis.json"), "r") as f:
      dta = json.load(f)
    kmer_analysis = KmerAnalysis(**dta)
  #If not, run the script then load the results
  else:
    %run Kmer_Analysis.ipynb
    with open(os.path.join("Standards","KmerAnalysis.json"), "r") as f:
      dta = json.load(f)
    kmer_analysis = KmerAnalysis(**dta)

  return rare_codon_analysis, codon_analysis, cpb_analysis, gc_analysis, kmer_analysis

def check_unit_tests():
  '''Check for scripts that can score proposed sequences'''
  print("Under construction")
  raise NotImplementedError

def degofdegen(aaseq):
    deg_dict = {'S': 6, 'L': 6, 'C': 2, 'W': 1, 'E': 2,
                'D': 2, 'P': 4, 'V': 4, 'N': 2, 'M': 1,
                'K': 2, 'Y': 2, 'I': 3, 'Q': 2, 'F': 2,
                'R': 6, 'T': 4, '*': 3, 'A': 4, 'G': 4, 'H': 2}
    deg = 1
    for aa in aaseq:
        deg *= deg_dict[aa]
    return deg


In [3]:
#Input AA seq
aaseq = input("Input an amino acid sequence")
fputr = ""
tputr = ""
desired_introns = []
promoter = ""
termintor = ""
finsulator = ""
tinsulator = ""
immismo_f = "CAGACCACCTGGCCAGGCTGCAGGAGCAGGGCCCAGGGCCTCCAGGAGCCAAGGCCTGGCCAGGGCTCCATGTTCCCCGAGGCCTTTCTGACTCAGGCTCCTGGCCTCTGGGAGCCTGGCCCTTCCAGGGAGCCACAGGGGATGGAACCTCCATCTGGAGCAGGGGAGAAGGCATGGAGTCCAGGCTTGTTTGAGGGGCCAAGGCTGAGCCCCCTCCAGCCCCTTCCCTGCTCTGCCATTAGGACCTGGCTCCTCCCAACTTCTATACACATCACATGGGCCGCGTCCCCCCACTTTGCAGGCTGAGAACATCTGACGTGACCTGGGCACCTGAGGGGCACCCGGCACAGCAGGAGCATGGACCTGGAGAACCTCTGTTCTTCCGTGCATCAAGGTCCGAGTCGTGGTTCTGCCCCTTAGCAGCTGCGTGACTCTGGCCAAGCTCCTCCCGGCAACAGGCCTCAGTTTCCATATCTGTGGATGGCACTAATCACGATGATGACCTGAGGAGACCGTTCTGCTGATGGAGTGAGCAGAATCACAAAGCTCTCAGGGCCTAGCCCGGGGCATGGCAGGTCCGCAGGCGTCCAAGGGCACAGCACAGATGTGGGAGAAAGGGCTGCGGGGCCCTCCCGCTCCAGCCGAGGACACGGGGCTCAGGGATGGAGCCCAATACTGTCCCAGAGCTCTGCTGCCCAGGTGCCTCTTGGGGAAACTGAGGCTCAGTCCCACAAAGGCATCAGTCCCACTGAAGACAGGGCCGAGTCCCCAGCCCCTCACCCCCCTCCAGGGGTCTGTGCCACCTCCAGGGAGCCGTGGCGGCTCCATGTCACCTGCGGGCAAGGGGCTGGTGTGGAAAGCCCCACGGCATGGTGGAAAGTCCGAAATTCTACAGGGGCCTCTTTGTTAAACCTCCATGCAAGAGGCTGGGTGACGCCTGCCCACAGGGGTCAGGCCCCAGCCCAATGACACAGGTGACCCGCAAAGACTGCAGGTCTTTTTTAAAAAGTGAAGTGCAGACTTGGATCTCGCTTTGCAGTTTTTAATTCCCCTTTAACCATCTCTGGGCTAAGGCATCCATTGACTCACTTAACCATCTGGGGAGGGAAAAGCCACCGGGTTGCAGGGGGTGACTTTGGGGCAGGATGGCCTCTGAAGGGGTGGGTTGCCCCTCCACACCTGTGGGTGTTTCTCGTTAGGTAGAACGAGAGACTTGGAAAAGAGACACAGACAAAGTATAGAGAAAGAAATTGGGGGACCAGGGGACCGGCGCTCAGCATACGGAGGACCCCCGCTGGCCTCTGAGTTCCCTTAGTATTTATTGATCATTTTTGGGTGTTTCTCGGAGAGGGGGATGTGGCAGGATCATAGGATAATAGTGGAGAGAAGGTCAGCAGGTAAGCACGTGAACAAAGGTCTCTGCATCATAAACAAGGTAAAGAATTAAGTGCTGTGCTTTAGATATGTATACACATAAACATCTCAATGCCTTAAGGAGCAGTATTGCTGCCTGCTTGTCCCACCTCCAGCCCTAAGGCAGTTTCCCCCTATCTCAGTAGATGGAATATACAATCGGGTTTTATACCGAGACATTCCATTGCCCAGGGACGGGCAGGAGACAGATGCCTTCCTCTTGTCTCAACTGCAACGAGGCGTTCCTTCCACTTTTACTAATCCTCCTCAGCACAGACCCTTTACGGTGTCGGGCTGGGGGACGGTCAGGTCTTTCCCTTCCCAGGAGGCCATATCTCAGGCTATCACATGGGGAGAAACCTTGGACAATACCTGGCTTTCCTAGGCAGGGGTCCCTGCGGCCTTCCGCAGTGTTTTGTGTCCCTGGGTACTTAAGATTAGGGAGTGGTGATGACTCTTAACGAGCATGCTGCCTTCAAGCATTTGTTTAATAAAGCACACCCTGCACAGCCCTTAATCCATTTAACCCTGAGTTGACACAGCACATGTCTCAGGGAGCACAGGGTTGGGGGTAGGGTTACAGATTAAAATGGAGTCTCTTATGTCTACTTTCTATGCAGACACATTAACAATCTGATCTCTCTTTCTTTTCCCCACAGCCTCAAAGGAGAGGAAAGCCAAGTTGCTGGGAGCAACGGTTACTGGTCATCCCAAAGCTGTGTGCGCTGGTTCCCTGGCACCCCAGGGGCTTTTATGCATGAACTGTGCAGCCAGTGGGCCCAAGGGTTGCCATTGAACTTGATGTTAGTGAGCTCTCCCTTCGTGAGACCCCTGCCCATGGATGAGCAAGGAGAATGGGGGTGTTTCAGGAGCAGACAGGGATCCTGTGCAAAGCTGGGCTTCTGTGCAACTTTCGCTTGCAGTTATTTAAATATTTTTGCTGTAAATACAGAACTGCAGGGGAGAGGGCAGGAAACCCAGCGAGCAGCAGCCCGGCCTGGCTGGGGACAGGATGTGTCTGTTGGAGCGGGGACCGGCAAGGCAGGCATGCAGGCAGGGGGCTTCCCTCTCGGGGTCTTCGGAAGGCGCAGTGCAGGGAGTGAGAGACGCCCAGGCCTGGGCAGCGAGAGGGCCCTGCTCCCCGCTCAAGGCTCCCAGGACATTCCACACAGGAGCCAGCCTGTTCCAACCCTGCACTGCCTGATACTCCACAGCCACCACTGAGATGGACCCCCTCCCCCTCAGCTCCCCATGACGTCCCCAGCACCTGACCCCAGGCTGACCACCTCGGCCACCACCTCTGGCCCCACCTTCCTCCCCAGGTTTGTTCCCAAACACCCCTAGAGTCTTTTCCGGAGCAATCTCAGACGTGCAGAACGCTCAGGGGGACCCTGGGCCTGAAGGCGGTAGGAGAAACCGAGACCGACCCCTGGTCTAGAGGAGAGGTGTCTGCCGCTCTCCATCCTGGACTCTGGCTGCCCAAAAGCAAACGGCACGCCTGCCCACGTGGGAATCTTGGTGGCGTCTCTCACTGCATCAGTTAGCCCCCAGCCCAGTGCTGCCTGGATTCAGGCTTCCTGGTACCTTCGTCCACCCTTCAGCCCTAATCCATGCTGGCCAGGTCATCTCGTTCTCTGGCTCAGTGGGAACACTCGTGCCCAGAGACCTCTCGTGGTTCTCTGCAGCCTGCACACAGGGCCTGCTTCCCTGGCTTGACCGCACAGACAAGGGGACTGTTACCAAGGCCTCCCACTGAGGCCACCCCCCCTCCAGGCCATGCCTGCGGGGCCACCACAGCCTCAGCATCATTGCAGGCCCCAGGCCTCTGCACCTGGTCTTGTTTTACTGGGGGCACTGTCCCCACTCACGTCCACCTGGGACCCTCGGCTCCTGTCCACTGCAGCTCCCCTCCCGGACACCTTCCCAGATGCCCCCGGAAGCTCCTGTCCAGGCCACAGCATCCCTCAGCCTCTGTCACTGGTCCTAGGAAGACCCTTTGGGAGCTCTCACTCAGGGCCACACTCAGGACCCCCGTTGTGGGGCTGGCCGCCTTCCTTCTAAAGCACCTGGAGGAAGGAAGGAGGGGGTCAATGCGAGCCTCAATCCCCAGGCGAGTGTGCCCCTTTTAAAGATGAGGGAACCGAGGCTCAGAGAAGGAAAGGACTTGCCTGGCGTCACACAGCTAGCCTAAGATGGTGAGGTCAGGAGCTCCCTGGCAACACCCAGGCTGCCGTACCGTCTCCTGCATCAGACTGAGCCTCCATCGGGCTCCTCCCACAGCCCCAGGCGGGCCCCTGAGTAGGGGCTCTCAGCTTGTGTGGAGGTCCCCCAGGAACACCACCCCGATCCAGCCTCCATGGAGGCTCTTGCCGGCCACTGGGAGGGGCCGGTGCACCCTGGGCAGCCCCTGCCAGGGCCCTGAGACCCGAGCCTCCCCGCCGAGGGCACCTGTCTCGGCTTTGCCCCATTCGAGCAGGGCCCTCGCCGAGGCAGGACAGGGCCACATTCGGAAGTGAGAGTTCTCTGAGTCCCGCACAGAGCGAGTCTCTGTCCCCAGCCCCCAAGGCAGCTGCCCTGGTGGGTGAGTCAGGCCAGGCCCGGAGACTTCCCGAGAGCGAGGGAGGGACAGCAGCGCCTCCATCACAGGGAAGTGTCCCTGCGGGAGGCCCTGGCCCTGATTGGGCGCCGGGGCGGAGCGGCCTTTGCTCTTTGCGTGGTCGCGGGGGTATAACAGCGGCGCGCGTGGCTCGCAGACCGGGGAGACGGGCGGGCGCACAGCCGGCGCGGAGGCCCCACAGCCCCGCCGGGACCCGAGGCCAAGCGAGGGGCTGCCAGTGTCCCGGGACCCACCGCGTCCGCCCCAGCCCCGGGTCCCCGCGCCCACCCCATGCACAGGAGGAGAAGCAGGAGCTGTCGGGAAGATCAGAAGCCAGTCATGGATGACCAGCGCGACCTTATCTCCAACAATGAGCAACTGCCCATGCTGGGCCGGCGCCCTGGGGCCCCGGAGAGGTATGTGTGAGCACCAGGAAAGGGCACACCGATCCTGGACTGCAGAGCCGTGCGCATGTGCTGGGAAGAAGCACCGGCCAGGAGTCACAGGAAAGAGGGATTCAGGCTCTGACTCTAACATTGACTTTTTGGGTGATTTTGAGCAAGTCTTTGGCCTGCTCTGGGCTCCGGCTCCCTCTCTTGTCAAATGAGGGGGTTCCTTTAAGAGCAGGAGTTTGTAACTTTCCTTGCATCACACCGCCTCCTTCTGAAAAGCAGAAGAGAGCATGGGTTCTCTCCCTAGAGTTTCACATGCTATGCAGGGGCTTTGAGATCCCACTAAAGCTGGGCTGTGAATGCTAGGTTAAAGGCCTTGGAACAGGCTGTGCCTGGTGGCTCCCGCCTGTAATCCCAGCACTTTGGGAGGCTGAGGCGGGCAGATCAGCTGAGGCCAGGAGTTCGAGACCAGCCTGGCCAACATGTTGAAACCCCATCTCTACTAAAAATACAAAAATTAGCCAGATGTGGTGGTGTACACCTGTAATCCCACCTACTTGGGAGGTTGAGACATGAGAATCACTTCAACCTGGGAGGCAGAGGTTGCAGTGAGCCAAGATTGTGCAACTGTACCCCACCCTGGGTGACACAGTGACACTCTGTCTCAAAAAAAATAAATAAATAAATAAAGACCTTGCACCAGATTTTGAAAATGTTCAAGATCCCAGTGGAGAAAGAGTTTCTAAAAGAGGATGGTAGAAGCAGTTCTTCACAGAGGAGTTCCCTCCTAATTACCACGGATCAAGGGAATGATAGTTGCAGGTGTCTGTAGAGTAGATGGAGACATCTGCCATCACAGGAGGGATTGTGAGATGTTGGGGGCAGGCAAGAAAGGAGGGACAGGCTCTGGTTAAATTTCTGGAGGGGGCTCAGGGTTGACGGGGGTCATGTCCTTGCCTGTGCCCCTCTCCCTTGCAGCCAGAGGCCAGCACAGCCATCCAGGGGACACCAGGCCCCCACTTCCCAGCCCCAAAATAGCCTCACCCATGTGTTTCCCTCAAAAAGGGCTAGAAGGCTATTACCCCAATGCCCCTGCTACCCCCAGTCTCCAAAAAGTAATTTGCGATCTACAGGGACTTACAGCTAGGTGTGATGTCAGCTGTTGCCAGGCAGAAAGGGGTTTGGGAGAACCCTGCAGAGATGTTATGACAAGCTATGTTCTGGGGAACAAGGTAACCTGCTGCAGACCATTAGAGCTGTGGCCTGTGAAAACCCCAGGGAATCCCATGGCAAGGAGGGAGACTGTTTTGTGAGGAATGGATTCAGGTCTAGGCCACGCGTGACCTTGGATAAATCACTGCCTCTCCCTGAGTCTCAGTAATAATATAATAGCCATAATAGTTCTCATCCACTGTGTACCTGTCACCAGGCCTCAGCCACTGTTAGTTAGTACTGTGATTATTTACATACGTAAAGGCTAAACTCAATTTTAATTCTTACAGTCACCCTATGAATCAGGTACTTTTGTTGTCCCTTTTTACAGAAGAGGAAATTGAGGTCATATTGGTAAGTGACTTACCCAAGTAATCATAGCTAGTGAGTGGTTGGAGGCAGAATTTGAACCAGGACTAGTCTGACATCAAAGTGTTAGCTCACTCCACCATTTGGCCCCCTTTCAGAGGTGCATTTTGTGGGAAGGGGAGGTGAGACCCCTCTGAGCTCAGACCACTCAGTGATTTACAGACCCAGGTGGGCTCAGTGTTCAGAGTGCATGTCCCCTTGTCTAATCCTTGTCTAGTTTTCCCTTTATTCAGTTACTGCTTCTAAAGCTGGCTATGTGTGCTTAGGCCATTCCAGACTGGAGCCTGTTTTGGAGGCTTTTGTGAATGCTTCTGCCTGGCAACATCTGTGTCACGATTCCTGCTAGCTCTGGTTGGCTCCTGACTTTGGCTTTTTTTTTTTTTTTTTTAATTGCAGAAGAAGGTAGTGTAAATGAGGAAATAAAGAGGGGGGTGACAAGAGTTGGCACTCTACCAGAACCAGGGGACAGGCGTTCAAGATGCATAAAGAGGAAGAAAAGCAGATGCTAAACTTAACTACGAACTCCCCTGAGAATTGCACTGAGTGTAATTTATTTATAGAGTGCTAACAATGTGCTAGGTGTGGAGCCAAGAGCTTTACCCTCATCATCTTATTTAATGTAACAGCCCTTGGAGCAGATGTTTTACCCGATTTTACTCACACAGAGGGTTAATCAACTTAGATGGTCCCAGAACTAACCGACTCCAGAGCCAGGATTCCAATCCAGGTTATGTGGTCTCGTCTTCAGGGTGAATCTCTAGGTTCTTAGGCCTCCTGAGGAACTGTGGTGCAGAGAGGGGAAGGTACTGTCCCATGTCGTGTGTGGGAGGTAGAGCTGGGGCTAGAATTGGGTGTGGGGTTGCAGAAGCACTGAACAGGTGGAGGATGAGGCTGGAGAGCCTGAACACAGGCTAGGCCACCCGTGTTCCTTGAGAAGGCCAAGAAGTAGAATTTGTTTTCCCTTCCTGTCTGGTCTCCCCACTTCCCCCCCGCCCCCCGCCCACCGTCGGGTTTTTGCTGTTTCCTTCCCGAATAACTCTCCCTTGGGCTGGGCCTTATCCCAGAAAGCACAGACTGGCACGCAGCAAGCCAATGAGAATGAGCCAGTCTGTGATGTCATGAGTTACCAGGCACAAGGGACTCCTGGAGCGTCCTAAACCATCCTTGCTGGTTGCTGGAGTAGGCAGGGTGCAGCCTCATCCTTGCAGTTCCCAATGGGTCTGTTTTAGTCCGTTTTGTGCCGCTTTAACAGAATACCACAGACTGCGTAATATACAAACAAACAAACAAACAAACAAAAAACACCCAGAAATTTATTTCTCACAGTTCTGGAGGGGCTTTCTTGTTTTTTTTTTTTTTTTTTGTTATGTTTTTTGAGACAGAGTCTCACTCTATCCCCCAGGATGGAATGCAGTGGTGTGATCTTGGCTCACAGCAACCTCTGCCTCCCAGGTTCAAGTGATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGACTACAGGTGCCCACCACTACGCCCGGCTAATTTTTTCTTTTTTTTTTGAGATGGAGCCTCACTCTGTTGCCCAGGCTGGAGTGCAATGGCATGATCTCAGCTCACTGCAACCTCCACCTTCCAGGTTCAAGGGATTCTCCTGCCTCAGCCTCCCGAGTAGCTGGGATTACAGGTGCTCGCCACCACGTCCGGCTAATTTTTGCATTTTTAGTAGAGACAGGGTTTCACCACGTTGGCCAGGCTGGTCTCGAACTCCTGACCTCAGGTGATCTACCTGTCTTGGCTTCCTAAAGTGCTGGGATTACAGGTGTGAGCCACCGCGCCCGGCCTAGCTTGCTTTCTTTTTTTTTTCTGAGACAGTCTCCTTCTGTCACCCAGGCTGGAGTGCAGTGACGTGATCTCGGCTCACTGCAACCTCCGCCTCCCGGGTTCAAGTGATTCTCCCACCTCAGCCTCCCAAGTAGCTGGGACTAGAGGTGCACGCCACCATGCCTGGCTAAGTTTTTTGTATTTTTTTGTAGAGACGGGGTTTCGCCATGTTGGCCAGACTGGTCTCGAACTCCTGGCCTCAAGTGATCCACCTGCCTTGGCCTCCCAAAGTGCTGGGATTACAGGCGTGAGTCACCACACCCAGCCTGGTGAGGGGCTTCCTTGATGTGTCCTCATATGGTGGAAGGCAGAAGGGCAAGAGAGAACCAACTCCTTCCATCAAGCCGTCTTACAGGGCACCTAATCCCATTCGTGAGGGAGAAGGCCTTGTGGCCCGATCACCTCCCAACACCTTCACATTGGCAACACCTGGACTAGGGTCTTAGTAGTCCCACCCCTGGGGTCGTCAAAGAACAGATCAGAAACTCCTGGGGTGTAGGGGCAGGATCCTTACATCCCCTGTACTGTAACATATGCATGCACCCCCCAGTCGCATCCAAGTCCCACAGATACAAGGGCTTGCTGAGGTCCGAGGGTCTGGATGGTGAGTCCAGGGAGCATGAGAAATTCATCCAGGCCACACAGGCTCAGGCACCTATACTCCTGCCCACCCCCTGCAGGATGGATATTACCATTTGAAATACTTTGGCGCTTGCAGTGAGATCTTGCTTAAAAGAAGAAACCCACCAAATTCTAGAATCGCAGAGGTGGGGGCACCTATGTGACAAATGAGGGAACAGAGGCACAGAGATTGGCAGGGACATGCTCAAAGACATGGAGAGCCTGGGCAGAGGATGGCCCCAGAGTGGGTGGTGCCTGGCCCTGCTTCAGCCTGACATTGTCCAAGCCTCCCCCAGCCCACTCCTAAAATGGCCAGGCCCCACTTCTTCCCAGGTGGGCTACATTTTCTGCTGTCCCGGTCCCACTTCCTTACCTTTTCTCCTTAAATCTGATTTCACTGTGAGGGTGGGAGGTTGGGCAGAGGGGTGAAAACCCTCTTCCCCTTACTTGGTAAGCACATACAATTTTTTTTGGTTTTGTCAATTAAAAAAAATAACATACCTGGTTGATCCTGCCAGCAGAAAAAAAAAACAAAACAAAAGGCTGGGCGCGGTGACTCACGCTTGTAATCCCAGCACTTTGGGAGGCTGAGGTGGGTGGATCACGAGATCAGGAGATCGAGACCACGGTGAAACCCCGTCTCTATCAAAAATACAAAATAATTAGCCAGGCGTGGTGGCGGGTGCCTGTAGTCCCAGCTACTCGGAGAGGCTGAGGAAGGAGAATGGCATAAACCCAGGAGGTGGAGGTTGCAGTGAGCCGAGATCGCGCCACTGCACTCCAGCCTGGGCGACAGAGCAAGACTCCGTCTCAAAAAACAAAACAAAACAAAACAAAACAAAAACCAAACCTGTTCCCCTTGAGGGCAGCCCTGATCATCAGGAGATCCGCACGGCAAAGGGAGCCTGGGAAGACCCCCCCACCTCAACCCCTCATCGCACAGGCAGGACCCTAAGGTCCAGAGAGGACAAAGAGCATGCCTCTTTGCTCTTCTCTGTTCAACTCAGGCCCGTGGCAGCTGCAGGGTCTCTGCCAGGGAAGACTTCAGGCACATCGATGGTTGGAAGGAGGGGTGGGGTGAGGTCTTCAAGCTGTGTTTCCATAGCCAGCTTACTGTTTGTGCTTCAAATGCATTGTTTATTAGGGGTGGCTTAGGGGTCTAATGGAGATGGGGGTATACATTTCCCCATTCCTAGCCAGCCCTTGGGGCAGCCACCACTGTGGATCCTCCTCTGACCTATCCTCCCCACCTCCCACAGCAAGTGCAGCCGCGGAGCCCTGTACACAGGCTTTTCCATCCTGGTGACTCTGCTCCTCGCTGGCCAGGCCACCACCGCCTACTTCCTGTACCAGCAGCAGGGCCGGCTGGACAAACTGACAGTCACCTCCCAGAACCTGCAGCTGGAGAACCTGCGCATGAAGCTTCCCAAGCGTGCGTGCACCCCTACATCCTGATACCCCCCACCTCCCACCATCCCTCAACTCAGAGACCCGCATCCCTGCACCCAGCTGGGCCCACTGTCCTCTCCCTCCGGTTTGGAATTCCAGCCCTTCCTCATCTGGGTCTGATACCCTCCTCCCTGGGCACCGGGGCCACACTTACCCTCGTTCCTGTCCCCACAGCTCCCAAGCCTGTGAGCAAGATGCGCATGGCCACCCCGCTGCTGATGCAGGCGCTGCCCATGGGAGCCCTGCCCCAGGGGGTAAGGACAGCCCCAGGGTGGTGGGAGGGGCAAGGTTATCCCGCCTGGATGGAGGACAGTGCCAAGGGGAGGGGCAGGGAAGAGAGCCCACCTGGGGAGGGGTCCTGACTGCTGCGGGAGGGACAGTGCCTGCCTCAGGAAGAATCGGGCTCCCCAGGTGTGGAGGGCACAGGTGAAGAGTCTCTTGGTGCCATCCCTGGGAGGAAGGCTCAGCCCTCTACAGTTTACAAAGTGCTTCTCATTTCCTATAGCATCTCACTGTCCTCTCCCATTCTCAAAGACCTTGCTATCATGCATTGACAATATTTATATTCACAATACTGTGCTGTGGACAAAACCCTGGGCAGGAAAGCTTATGCCAGTTTGACCAATGAGGACATTGAGGCAGGAAGCTAAAGTGACTTGCTCGAGCTCTCATGTTTGGAGGTGGCAGAGATGGAACCATTGACCAAGTGCCTGCGATTCCAGTCTCTTACCTAGATCCCAGCAACCGGCTCCTGCTCCATACCCCCTGCTCCAGGGACCAGCTCTGGTAACCTTCTGTTACTTCCTCCCACAGCCCATGCAGAATGCCACCAAGTATGGCAACATGACAGAGGACCATGTGATGCACCTGCTCCAGGTGAGTGCAGGGAGCTAGCTGGGTGGTCCTGCCTGCCCACCCAGGACCCTGGCCGGGCCAAGCTCCAAGGCCTGTATACCTGGCTCATGGCAGACTTTCAACATGTGCTCTCTGGATTACTGAATACAAGGGTGACCCTTAAATGTTATATGTAGTCTGGCCTCTGCATTTTTGAGATAAACAGGCTCGGCTGGGTGCAGTGGCTCATGCCTGTAATCTCAGCACTTTGGGAGGCCTAGGCAGGCAGATCACCTGAGGCCAGGAGTTCGAGACCAGCCTGGCCAACATAGTGAAAGTCTCTACTAAAAATACAAAAATTAGCCGGATGTGGTGGCAGTCACCTGTAATCCCAGCTACTCATGAGGTTGAGGCAGGAGAATCGCTTTAACCCGGGAAGTGGAGGTTGCAGTGAGCCGAGATCATCACGCCACTGCACTCCAGCCTGGGTGACAGAGTGAGACTCCATTTCAAAAAATAAAATAAAATAAAAATAAAAAAAGAAACTGAGGCTCAAAGTTTGGAAGTGACTTGCTCAAGACACCCTTTTTTTCTCCCCAAATTTGTACTGCGTGTCTGTCCTATGCCAGCACTGTGCAAGGCTCTGGGCCCACAATGGGGAGCAATTGGACCCAGCTCCTGAAGCCATGGGGATCCCAGGCTAGTGTGGGAGACAGACAAGTGACCAGGTGATGACAGAAGTGCAGGGGGCTTTGTGAAGAAAGAGGCAGAGGACTCAACCTAGCTGGGGGCAGTCAAGGTTGGGTCCTCACCTGCTGGAGCAGGTAAGACCTGAAAGCTGAGCAGCCATCCACAGGGAGGACACTGGGGCTTGTTGGTAGAGCTGGCCTGGAATCTGGAACTCGACTCGCAACGCTGCAGTCTCGTAGCTGAACAGCTGGACTACCCTTACCCGTACCCTCCCCTCCCCTCCCATCATGGCTGAAGTCCAGGGTTTAGGCTTGGTGTAGCTCTGGTCCATTCCTCAGAATGGAAGACCAAGGGAACGTGGGCTACACATTTCACACTGCCTGCCAGGGAGCTCTTCTTGAACCATCCTTCCTGCCCTGCTACCCTGTAGAATGCTGACCCCCTGAAGGTGTACCCGCCACTGAAGGGGAGCTTCCCGGAGAACCTGAGACACCTTAAGAACACCATGGAGACCATAGACTGGAAGGTCAGCAGGTTTCCCTGCATGGGAACTCTCTCTTTCCTCTGGTGTCTAGGGCAGGGCTAGGAGAGGTGGGGTGAGGGTGGGCTGGGGAAGCCATTCTCAGGAAGCTGAAGGGGTTTACCAGCACTTCCAAAACCTGGATGCTGCAGAGTATGTGGGGTTCAGCCCCAGGGGTCTTTAAGAGGGAACCAGGCTGCAGCTGGACCCGGGTGTTGGGGCCCTATTTGGCCCTGCATGTTTCTGTCCCCAGGGGCAAAGCCAGGCAGTGTAGGGGCTTGGTGGTGGCCATCGAACCTGACCTCCACCTCTATCCGTATTAGGTCTTTGAGAGCTGGATGCACCATTGGCTCCTGTTTGAAATGAGCAGGCACTCCTTGGAGCAAAAGCCCACTGACGCTCCACCGAAAGGTACAGGGAGTGGGAGCTTTAGCGTGCCAGGGCTTCTGGACCCTCGGGGCTCTCCTGAAGCTGCTGAGGCCGGGGCCTCCAGCACTCCCTGGTCCCAGCACCGCGGAATCTCCCATCCTCTCAGCTCTCACTTCTTTCTCACTTCTCTCTCTCTCCTGTCTGTTCCTTCTTGGTTTGGCTGTCCCCCTCCCCTGACCCACCCCCATCTTGTCTAAGGGTTCCTAAGGGCCCACAGAGGCCTGTCACCACAGGGTACAGGTGACCTCTCTCATAGAGGATGGTACAGCACAGGGTGCCTGGGGTAGAACCTGCCCGAAACACTGCAGAAAGGAATCCTTGTAATGACCTTGCCCCAGTGCTGCCCGCAATCCAGTGAGGGCGCCAAGGTCACAGCTGCTGCCAAGAGAGCCTTGGGCGTTTCCCACCTCATGGACAATGCAGACTAGGATGTTTTTAGACCCAAAGACAGAGTGCTGTTTCTATCCCGGTGCTGTCTCTAATTTGCTATGTAACTTTGGACAAGTCCCCTTCCCTCTAGGATTCAGGGTCCTGAAGTAGAAGGTCAAAGGGCCACCCTGCCTGGGGCCTCAGTTTCTGCATCAGATTCATAGAAGGCACCTTACATGCTATCTCCAACTCCTAGCTGATGCTTAAACCCCTCTAAGACATCTCCAACAAACGGTAAACCCCATTTCTACTTGAGAACTCCCAGTAACAGGAAGCTTAGACTTACCAAGGTGCCCATTGCTTTTCTCTAGAATCAGAATCGCAAGTGCAATTCCAAACTGTAATGGTGTTTTTTTGTTTGATTGTTTGTTTTTGAGATAGAGTCTGGCTCTGTCGCCCAGGCTGGAGTGCAGTGGTGCGATCTCAGCTCACTGCAACCTCAGCCTCCCGGGTTCAAGCAATTCTCCTGCCTCGGCCTCCCGAGTAGCTGGGATTATAGGCATGTGCCACCACGCCCAGCTAATTTTTGTATTTTTAGTAGAGACAGGGTTTCACCATGTTGGCCAGGCTGGTCTTGAACTCCTGACCTCAAGTGATCCGCCCACCTCGGCCTCCCAAAGTGCTAAGATTACAGGCATGAGCCACTGTGCCTGGCCCAAAACTGTAATGGGCTTTGAGTGTCAGAAGAAACCATCAACTTCTGAGGTGAATTGGACACATGGCCATTCACTTCCTTTTTGATCTCAGACCTTGTTGGTCTAGGCCTCAGTTTTCCCATCCGTGTGATGGCTGGAGTGAGTAAAGCCACTTGGGAAGAAGGCATTAAGCCCACAGCAGTGGTGTGTGGGTCTTTAGCTCTGCTCAGACCCTGGTTCAGAGCTCACTCACTCACTGTGTCCTCATCATGCCTGTCGCTTCAGTACTGACCAAGTGCCAGGAAGAGGTCAGCCACATCCCTGCTGTCCACCCGGGTTCATTCAGGCCCAAGTGCGACGAGAACGGCAACTATCTGCCACTCCAGTGCTATGGGAGCATCGGCTACTGCTGGTGTGTCTTCCCCAACGGCACGGAGGTCCCCAACACCAGAAGCCGCGGGCACCATAACTGCAGTGGTAAGCAGTGGCACTGTGCCAGTGTCAGAGGACCAGGAAGGACTAGGAAGGTTGAGGGGCAAGAGGTCCCCTCTGAAGCACATGGGACCAGGACACCCAGGATGGCAGCTCCTGGGGGCAGTGACGTAGTCATGCGTCCAGCCCTTTATCCATCCACCCACCTGTGCATTCCTATTTGTCCATTTATCATCTCTCCTCTTACATGCACTCATATTTGTTCACTTATCCGTCAGTCCTTTCATCTGTGAATTTCATCCATCCACGCGCCATCCTCTAGCATCCAGGAGTCCTACAGACCCACCCATCTCATTTCATCACCCCCTTCTCACTCGAGCCCCCATTACACCTCTTTGTTGCTGCTTGCAGCTGTCCTTCCCTGGGTACCTGCTTTCCCAGGCACTAAGCCTGTGGCTAGGTGGGAAGCACTGCCCTCAGGCATCTTGGGTCAGGTAGGCTGCACTTCAAGTGACAAACGGACTTGCTGCTCCTTTGCAGAGTCACTGGAACTGGAGGACCCGTCTTCTGGGCTGGGTGTGACCAAGCAGGATCTGGGCCCAGGTAAGGGCCTTGCAGAGGGGCATCTGGTCACCAGCAGCTCATCCCCAGCAGGGCCAGCTCCTTTGTGGGCAGGTGAAGGAGTGTGACGCTGGGCCACTCTCCAACATTCCTGGGATGTCCATTTCACAGACGAAGGAACAGGGTTGGGGGGCTGTGGGGAGTTACACAAATCCATGATGGTCATTATTGGACCTGAGCAGGGGGCAGGGGAGGGTGGACAGTCCTTAACTGCTCTGCAGGTCCAGGATGTTAGAAAGGGGCAGGGACAACAAATGGGTGACCCCAACCTCAACCTGCTGCTTCTCTCTCCAGTCCCCATGTGA"
immismo_t = "GAGCAGCAGAGGCGGTCTTCAACATCCTGCCAGCCCCACACAGCTACAGCTTTCTTGCTCCCTTCAGCCCCCAGCCCCTCCCCCATCTCCCACCCTGTACCTCATCCCATGAGACCCTGGTGCCTGGCTCTTTCGTCACCCTTGGACAAGACAAACCAAGTCGGAACAGCAGATAACAATGCAGCAAGGCCCTGCTGCCCAATCTCCATCTGTCAACAGGGGCGTGAGGTCCCAGGAAGTGGCCAAAAGCTAGACAGATCCCCGTTCCTGACATCACAGCAGCCTCCAACACAAGGCTCCAAGACCTAGGCTCATGGACGAGATGGGAAGGCACAGGGAGAAGGGATAACCCTACACCCAGACCCCAGGCTGGACATGCTGACTGTCCTCTCCCCTCCAGCCTTTGGCCTTGGCTTTTCTAGCCTATTTACCTGCAGGCTGAGCCACTCTCTTCCCTTTCCCCAGCATCACTCCCCAAGGAAGAGCCAATGTTTTCCACCCATAATCCTTTCTGCCGACCCCTAGTTCCCTCTGCTCAGCCAAGCTTGTTATCAGCTTTCAGGGCCATGGTTCACATTAGAATAAAAGGTAGTAATTAGAA"
syngene = SyntheticGene(aaseq, "GeneRider optimization for Immismo-T1DM", [], degofdegen(aaseq), [])
rc, c, cpb, gc, kmer = standards_check()
aobjs = {'RareCodonAnalysis': rc, 'CodonAnalysis': c, 'CodonPairBiasAnalysis': cpb, 'GCAnalysis': gc, 'KmerAnalysis': kmer}

Input an amino acid sequenceGAGCAGCAGAGGCGGTCTTCAACATCCTGCCAGCCCCACACAGCTACAGCTTTCTTGCTCCCTTCAGCCCCCAGCCCCTCCCCCATCTCCCACCCTGTACCTCATCCCATGAGACCCTGGTGCCTGGCTCTTTCGTCACCCTTGGACAAGACAAACCAAGTCGGAACAGCAGATAACAATGCAGCAAGGCCCTGCTGCCCAATCTCCATCTGTCAACAGGGGCGTGAGGTCCCAGGAAGTGGCCAAAAGCTAGACAGATCCCCGTTCCTGACATCACAGCAGCCTCCAACACAAGGCTCCAAGACCTAGGCTCATGGACGAGATGGGAAGGCACAGGGAGAAGGGATAACCCTACACCCAGACCCCAGGCTGGACATGCTGACTGTCCTCTCCCCTCCAGCCTTTGGCCTTGGCTTTTCTAGCCTATTTACCTGCAGGCTGAGCCACTCTCTTCCCTTTCCCCAGCATCACTCCCCAAGGAAGAGCCAATGTTTTCCACCCATAATCCTTTCTGCCGACCCCTAGTTCCCTCTGCTCAGCCAAGCTTGTTATCAGCTTTCAGGGCCATGGTTCACATTAGAATAAAAGGTAGTAATTAGAA


KeyboardInterrupt: 

#Generate seed solutions from the codon vectors

In [ ]:
def generate_seed(syntheticgene: SyntheticGene):
  '''Generate a seed solution from a synthetic gene'''
  cvec = generate_codon_vec(syntheticgene.AASeq)
  sol = []
  for i in range(0, len(syntheticgene.AASeq)):
    sol.append(random.choice(cvec[i]))
  return sol


nseeds = 2**(len(syngene.AASeq)//4)
print(nseeds)
seeds = [generate_seed(syngene) for x in range(0,nseeds)]
print(seeds[0])

#Multi-seed greedy algorithm with GA architecture

In [ ]:
def dist_from_optimal(codons: list, bobj: CodonAnalysis):
  '''Figure out what the optimal score would be for a stream of codons'''
  aaCodons = {
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
    'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'C': ['TGT', 'TGC'],
    'W': ['TGG'],
    'E': ['GAA', 'GAG'],
    'D': ['GAT', 'GAC'],
    'P': ['CCT', 'CCC', 'CCA', 'CCG'],
    'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'N': ['AAT', 'AAC'],
    'M': ['ATG'],
    'K': ['AAA', 'AAG'],
    'Y': ['TAT', 'TAC'],
    'I': ['ATT', 'ATC', 'ATA'],
    'Q': ['CAA', 'CAG'],
    'F': ['TTT', 'TTC'],
    'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    '*': ['TAA', 'TAG', 'TGA'],
    'A': ['GCT', 'GCC', 'GCA', 'GCG'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG'],
    'H': ['CAT', 'CAC']}
  opt_score = 0
  for codon in codons:
    assert len(codon) == 3
    #Figure out what aa the codon encodes
    encaa = ''
    for aa in aaCodons.keys():
      if codon in aaCodons[aa]:
        encaa = aa
        break
    assert encaa != ''
    cods = aaCodons[encaa]
    scores = {c:bobj.codonFreqsLit[c] for c in cods}
    opt_score += scores[max(scores, key=scores.get)]

  return opt_score/len(codons)

def calculate_change_vector(sol: list, syngene: SyntheticGene, analysis_objects: dict):
  '''Calculate the change vector for a given solution'''
  changevecs = {}
  #For Immismo only.
  locvec = [] #CD74 end exon goes 'S' 'T' 'T' 'T*'
  for i in range(0, 12):
    locvec.append('T')
  for i in range(12, len(sol)):
    locvec.append('I')


  assert 'RareCodonAnalysis' in analysis_objects.keys(), "RareCodonAnalysis not in analysis_objects"
  winsize = 15
  acobj = analysis_objects['RareCodonAnalysis']
  rarelist = ['GCG', 'CCG', 'CGT', 'CGC', 'TCG', 'ACG']
  rvec = [1 if c in rarelist else 0 for c in sol]
  wins = [rvec[i:i+winsize] for i in range(0, len(rvec)-winsize)]
  totwins = sum([acobj.rare_codon_windows[key] for key in acobj.rare_codon_windows.keys()])
  oddscores = {}
  for key in acobj.rare_codon_windows.keys():
    if acobj.rare_codon_windows[key] == 0:
      oddscores[key] = float.fromhex('0x1.fffffffffffffp+1023')
    else:
      oddscores[key] = totwins/acobj.rare_codon_windows[key]
  scorewins = []
  for win in wins:
    score = sum(win)
    scorewins.append(oddscores[score])
  scorevec = []
  for i in range(0, len(sol)):
    #Figure out which windows overlay at this position
    relevant_windows = []
    h = winsize // 2
    if i < winsize:
      relevant_windows = scorewins[0:i]
    elif i > len(sol)-winsize:
      relevant_windows = scorewins[i:len(sol)]
    else:
      relevant_windows = scorewins[i-h:i+h]
    try:
      val = sum(relevant_windows)/len(relevant_windows)
    except:
      val = float.fromhex('0x1.fffffffffffffp+1023')
    scorevec.append(val)
  assert len(scorevec) == len(rvec), "Scorevec and sol are not the same length"
  overall_rc = sum(rvec)/len(rvec)
  distrib = acobj.usagePerGene
  zscore = (overall_rc - distrib.mean())/distrib.std() #Given the right-side distribution of rare codon freq for all genes in the human genome, get a z score for our observation overall_rc
  zscore = zscore**2
  rarechangevec = []
  for y in range(0, len(rvec)):
    if scorevec[y] == float.fromhex('0x1.fffffffffffffp+1023'):
      v = float.fromhex('0x1.fffffffffffffp+1023')
    else:
      v = scorevec[y]*rvec[y]*zscore
    rarechangevec.append(v)
  changevecs['RareCodons'] = rarechangevec

  del acobj
  del rvec
  del wins
  del totwins
  del oddscores
  del scorewins
  del scorevec
  del overall_rc
  del distrib
  del zscore
  del rarechangevec

  #TODO: Currently not location-aware
  assert 'CodonAnalysis' in analysis_objects.keys(), "CodonAnalysis not in analysis_objects"
  acobj = analysis_objects['CodonAnalysis']

  cuvec = [acobj.codonFreqsLit[codon] for codon in sol]
  cuwins = [sol[i:i+acobj.windowsize-1] for i in range(0, len(sol)-acobj.windowsize)]
  cuwinscores = []
  cudists = []
  for win in cuwins:
    cuwinscores.append(sum([acobj.codonFreqsLit[codon] for codon in win])/acobj.windowsize)
    cudists.append(dist_from_optimal(win, acobj))
  dist_zs = []
  score_zs = []
  for i in range(0, len(sol)):
    #Figure out which windows overlay at this position
    relevant_windows_d = []
    relevant_windows_s = []
    h = acobj.windowsize // 2
    if i < winsize:
      relevant_windows_d = cudists[0:i]
      relevant_windows_s = cuwinscores[0:i]
    elif i > len(sol)-winsize:
      relevant_windows_d = cudists[i:len(sol)]
      relevant_windows_s = cuwinscores[i:len(sol)]
    else:
      relevant_windows_d = cudists[i-h:i+h]
      relevant_windows_s = cuwinscores[i-h:i+h]
    val_d = sum(relevant_windows_d)/len(relevant_windows_d)
    val_s = sum(relevant_windows_s)/len(relevant_windows_s)
    z_d = (val_d - acobj.windowdistancesfromoptimal.mean())/acobj.windowdistancesfromoptimal.std()
    dist_zs.append(z_d**2)
    z_s = (val_s - acobj.windowscores.mean())/acobj.windowscores.std()
    score_zs.append(z_s**2)
  m = acobj.codonUsageScoreByGene.mean()
  s = acobj.codonUsageScoreByGene.std()
  straight_z = [(o-m)/(2*s) for o in cuvec]
  assert len(straight_z) == len(sol), "straight_z and sol are not the same length"
  assert len(score_zs) == len(sol), "score_zs and sol are not the same length"
  assert len(dist_zs) == len(sol), "dist_zs and sol are not the same length"
  codonusagechangevec = [straight_z[i]*score_zs[i]+ straight_z[i]*dist_zs[i] for i in range(0, len(sol))]
  changevecs['CodonUsage'] = codonusagechangevec

  del acobj
  del cuwins
  del cuwinscores
  del cudists
  del score_zs
  del dist_zs
  del straight_z
  del codonusagechangevec

  assert 'CodonPairBiasAnalysis' in analysis_objects.keys(), "CodonPairBiasAnalysis not in analysis_objects"
  acobj = analysis_objects['CodonPairBiasAnalysis']
  cpbs = [sol[i] + sol[i+1] for i in range(0, len(sol)-1)]
  cpbscorevec = [acobj.cpb_lit[cpb] for cpb in cpbs]
  wins = [sol[i:i+acobj.windowlength-1] for i in range(0, len(sol)-acobj.windowlength)]
  cpbwinscores = []
  for win in wins:
    wincpbs = [win[i] + win[i+1] for i in range(0, len(win)-1)]
    cpbwinscores.append(sum([acobj.cpb_lit[cpb] for cpb in wincpbs])/acobj.windowlength)
  e = acobj.cpbPerWindow.mean()
  s = acobj.cpbPerWindow.std()
  winzs = [((o-e)/s)**2 for o in cpbwinscores]
  winchangevec = []
  for i in range(0, len(sol)):
    #Figure out relevant windows
    relevant_windows = []
    h = acobj.windowlength // 2
    if i < winsize:
      relevant_windows = winzs[0:i]
    elif i > len(sol)-winsize:
      relevant_windows = winzs[i:len(sol)]
    else:
      relevant_windows = winzs[i-h:i+h]
    val = sum(relevant_windows)/len(relevant_windows)
    winchangevec.append(val)
  cpbchangevec = []
  maxcpb = max([acobj.cpb_lit[key] for key in acobj.cpb_lit.keys()])
  for i in range(0, len(sol)):
    if i == 0:
      cpbscore_ave = cpbscorevec[i]
    elif i == len(sol)-1: # Fixed index here
      cpbscore_ave = cpbscorevec[-1]
    else:
      cpbscore_ave = (cpbscorevec[i-1] + cpbscorevec[i])/2


  compoundvec = [winchangevec[i]*cpbchangevec[i] for i in range(0, len(sol)-1)] # Fixed index here
  changevecs['CodonPairBias'] = compoundvec

  del acobj
  del cpbs
  del cpbscorevec
  del wins
  del cpbwinscores
  del winzs
  del winchangevec
  del maxcpb
  # cpbscore_ave is overwritten in each iteration, no need to delete
  del cpbchangevec
  del compoundvec


  assert 'GCAnalysis' in analysis_objects.keys(), "GCAnalysis not in analysis_objects"
  acobj = analysis_objects['GCAnalysis']
  from numpy import mean # Import mean function
  gc1mean = mean([0 if sol[i][0] in ['G', 'C'] else 1 for i in range(0, len(sol))])
  gc2mean = mean([0 if sol[i][1] in ['G', 'C'] else 1 for i in range(0, len(sol))])
  gc3mean = mean([0 if sol[i][2] in ['G', 'C'] else 1 for i in range(0, len(sol))])
  m1l = acobj.taggedGC1['ExonL50'].mean()
  s1l = acobj.taggedGC1['ExonL50'].std()
  m1e = acobj.taggedGC1['Exon'].mean()
  s1e = acobj.taggedGC1['Exon'].std()
  m1r = acobj.taggedGC1['ExonR50'].mean()
  s1r = acobj.taggedGC1['ExonR50'].std()
  m1s = acobj.taggedGC1['Splice'].mean()
  s1s = acobj.taggedGC1['Splice'].std()
  m2l = acobj.taggedGC2['ExonL50'].mean()
  s2l = acobj.taggedGC2['ExonL50'].std()
  m2e = acobj.taggedGC2['Exon'].mean()
  s2e = acobj.taggedGC2['Exon'].std()
  m2r = acobj.taggedGC2['ExonR50'].mean()
  s2r = acobj.taggedGC2['ExonR50'].std()
  m2s = acobj.taggedGC2['Splice'].mean()
  s2s = acobj.taggedGC2['Splice'].std()
  m3l = acobj.taggedGC3['ExonL50'].mean()
  s3l = acobj.taggedGC3['ExonL50'].std()
  m3e = acobj.taggedGC3['Exon'].mean()
  s3e = acobj.taggedGC3['Exon'].std()
  m3r = acobj.taggedGC3['ExonR50'].mean()
  s3r = acobj.taggedGC3['ExonR50'].std()
  m3s = acobj.taggedGC3['Splice'].mean()
  s3s = acobj.taggedGC3['Splice'].std()
  overall_gc1_z_l = (gc1mean - m1l)/s1l
  overall_gc2_z_l = (gc2mean - m2l)/s2l
  overall_gc3_z_l = (gc3mean - m3l)/s3l
  overall_gc1_z_e = (gc1mean - m1e)/s1e
  overall_gc2_z_e = (gc2mean - m2e)/s2e
  overall_gc3_z_e = (gc3mean - m3e)/s3e
  overall_gc1_z_r = (gc1mean - m1r)/s1r
  overall_gc2_z_r = (gc2mean - m2r)/s2r
  overall_gc3_z_r = (gc3mean - m3r)/s3r

  #For each codon, determine if we can actually change gc1 by swapping codons
  codon_vector = generate_codon_vec(syngene.AASeq)
  gc1io = []
  gc2io = []
  gc3io = []
  for i in range(0, len(sol)):
    aa = syngene.AASeq[i]
    curr_cod = sol[i]
    all_cods = codon_vector[i]
    if all(c[0] in ['G', 'C'] for c in all_cods):
      gc1io.append(0)
    else:
      gc1io.append(1)
    if all(c[1] in ['G', 'C'] for c in all_cods):
      gc2io.append(0)
    else:
      gc2io.append(1)
    if all(c[2] in ['G', 'C'] for c in all_cods):
      gc3io.append(0)
    else:
      gc3io.append(1)
  gc1changevec = []
  gc2changevec = []
  gc3changevec = []
  for i in range(0, len(sol)):
    if locvec[i] == 'T':
      gc1changevec.append(gc1io[i]*overall_gc1_z_l**2) # Removed list comprehension and fixed syntax
      gc2changevec.append(gc2io[i]*overall_gc2_z_l**2) # Removed list comprehension and fixed syntax
      gc3changevec.append(gc3io[i]*overall_gc3_z_l**2) # Removed list comprehension and fixed syntax
    elif locvec[i] == 'I':
      gc1changevec.append(gc1io[i]*overall_gc1_z_e**2) # Removed list comprehension and fixed syntax
      gc2changevec.append(gc2io[i]*overall_gc2_z_e**2) # Removed list comprehension and fixed syntax
      gc3changevec.append(gc3io[i]*overall_gc3_z_e**2) # Removed list comprehension and fixed syntax
    else:
      raise NotImplementedError

  #Combine the codons from sol into one continuous strand
  continuous_sol = ''
  for codon in sol:
    continuous_sol += codon
  windows = [continuous_sol[i:i+acobj.windowsize-1] for i in range(0, len(continuous_sol)-acobj.windowsize)]
  windows_zs = []
  for i in range(0,len(windows)):
    indexes = range(i, i+acobj.windowsize-1)
    windowloc = [locvec[index//3] for index in indexes] # Fixed index
    #Get most popular letter from the list of letters
    windowloc = max(set(windowloc), key=windowloc.count)
    window_gc = mean([0 if windows[i][j] in ['G', 'C'] else 1 for j in range(0, len(windows[i]))]) # Added mean()
    if windowloc == 'T':
      e = acobj.windows['ExonL50'].mean()
      s = acobj.windows['ExonL50'].std()
      windows_zs.append(((window_gc - e)/s)**2)
    elif windowloc == 'I':
      e = acobj.windows['Exon'].mean()
      s = acobj.windows['Exon'].std()
      windows_zs.append(((window_gc - e)/s)**2)
    else:
      raise NotImplementedError
  assert len(windows_zs) == len(windows), "windows_zs and windows are not the same length"
  gcchangevec_cont = []
  for i in range(0, len(continuous_sol)):
    relevant_windows = []
    h = acobj.windowsize // 2
    if i < winsize:
      relevant_windows = windows_zs[0:i]
    elif i > len(continuous_sol)-winsize: # Fixed index
      relevant_windows = windows_zs[i:len(windows_zs)] # Fixed index
    else:
      relevant_windows = windows_zs[i-h:i+h]
    val = sum(relevant_windows)/len(relevant_windows)
    gcchangevec_cont.append(val)
  gcchangevec_cods = [mean(gcchangevec_cont[i:i+3]) for i in range(0, len(gcchangevec_cont), 3)]
  assert len(gcchangevec_cods) == len(sol), "gcchangevec_cods and sol are not the same length"
  compoundvec = [gcchangevec_cods[i]*(gc1changevec[i]+gc2changevec[i]+gc3changevec[i]) for i in range(0, len(sol))]
  changevecs['GC'] = compoundvec

  del acobj
  del gc1mean, gc2mean, gc3mean
  del m1l, s1l, m1e, s1e, m1r, s1r, m1s, s1s
  del m2l, s2l, m2e, s2e, m2r, s2r, m2s, s2s
  del m3l, s3l, m3e, s3e, m3r, s3r, m3s, s3s
  del overall_gc1_z_l, overall_gc2_z_l, overall_gc3_z_l
  del overall_gc1_z_e, overall_gc2_z_e, overall_gc3_z_e
  del overall_gc1_z_r, overall_gc2_z_r, overall_gc3_z_r
  del gcchangevec_cont, gcchangevec_cods
  del gc1io, gc2io, gc3io
  del gc1changevec, gc2changevec, gc3changevec
  del compoundvec

  assert 'KmerAnalysis' in analysis_objects.keys(), "KmerAnalysis not in analysis_objects"
  acobj = analysis_objects['KmerAnalysis']
  krange = [int(g) for g in acobj.kmer_dict.keys()]
  kmerchangevecs = []
  continuoussol = ''
  locationstring = ''
  for g in range(0, len(sol) -1):
    assert len(sol[g]) == 3, "sol[g] is not a codon"
    continuoussol += sol[g]
    locationstring += locvec[g]
    locationstring += locvec[g]
    locationstring += locvec[g]
  for k in krange:
    solkmerswins = [continuoussol[i:i+k] for i in range(0, len(continuoussol)-k-1)]
    sollocwins = [locationstring[i:i+k] for i in range(0, len(locationstring)-k-1)]
    assert len(solkmerswins) == len(sollocwins), "solkmerswins and sollocwins are not the same length"
    solkmerswinscores = []
    for f in range(0, len(solkmerswins)):
      win = solkmerswins[f]
      loc = sollocwins[f]
      loc = [loc[p] for p in range(0,k)] # Fixed index
      assert len(win) == len(loc), "win and loc are not the same length"
      loc = max(set(loc), key=loc.count)
      assert len(loc) == 1, 'location consensus not reached'
      match loc:
        case 'T':
          solkmerswinscores.append(acobj.kmer_dict[str(k)][win]['ExonL50']) # Fixed key to string
        case 'I':
          solkmerswinscores.append(acobj.kmer_dict[str(k)][win]['Exon']) # Fixed key to string
        case _:
          raise NotImplementedError
    overall_kmer_score_standardized = sum(solkmerswinscores)/len(solkmerswins) # Fixed variable name
    winscores_to_continuous = []
    #Average the relevant window scores for each position along the continuous solution
    for i in range(0, len(continuoussol)):
      relevant_windows = []
      if k % 2 == 0:
        h = k // 2 #- 1
      else:
        h = k // 2
      if i < winsize:
        relevant_windows = solkmerswinscores[0:i]
      elif i > len(continuoussol)-winsize: # Fixed index
        relevant_windows = solkmerswinscores[i:len(solkmerswinscores)] # Fixed index
      else:
        relevant_windows = solkmerswinscores[i-h:i+h]
      if len(relevant_windows) > 0: # Added check for empty list
        val = sum(relevant_windows)/len(relevant_windows)
      else:
        val = 0 # Or some other default value
      winscores_to_continuous.append(val)
    assert len(winscores_to_continuous) == len(continuoussol), "winscores_to_continuous and continuoussol are not the same length"
    winscores_to_continuous = [winscores_to_continuous[i]*overall_kmer_score_standardized for i in range(0, len(winscores_to_continuous))]
    assert len(winscores_to_continuous) == len(continuoussol), "winscores_to_continuous and continuoussol are not the same length"
    winscores_codons = [winscores_to_continuous[i:i+3] for i in range(0, len(winscores_to_continuous), 3)]
    assert len(winscores_codons) == len(sol), "winscores_codons and sol are not the same length"
    kmerchangevecs.append(winscores_codons)
  final_kmervec = []
  for i in range(0, len(sol)):
    final_kmervec.append(sum(kmerchangevecs[j][i] for j in range(0, len(kmerchangevecs)))) # Fixed index
  changevecs['Kmer'] = final_kmervec

  del acobj
  del krange
  del kmerchangevecs
  del continuoussol
  del locationstring
  del solkmerswins
  del sollocwins
  del solkmerswinscores
  del winscores_to_continuous
  del winscores_codons
  del final_kmervec






  for key in changevecs.keys():
    print(key)
    print(changevecs[key][0:25])
    print("_________________________")

  return changevecs

def kill_off(pop: list, weights, percent_cut = 30):
  '''Takes scored vectors, kills off the % of them that have the highest need for change'''
  assert all(type(p) == ProposedSolution for p in pop), "pop is not a list of ProposedSolution objects"
  total_num = 0
  for p in pop:
    total_num += p.number
  num_to_kill = total_num * percent_cut // 100
  scores = [score_changevec(p.codons, p.changevecs, weights) for p in pop]
  while num_to_kill > 0:
    #Choose a random index to kill, weighed by values of scores
    index_to_kill = random.choices(range(0, len(pop)), weights=scores)[0]
    #Check to see what the number is of pop[index_to_kill]
    if pop[index_to_kill].number > 1:
      pop[index_to_kill].number -= 1
      num_to_kill -= 1
    else:
      pass
  return pop




def score_changevec(sol: list, changevecs: dict, weights: dict):
  '''Take in a change vector and return a score'''
  score = 0
  for key in changevecs.keys():
    score += weights[key]*sum(changevecs[key])
  return score


def replicate_and_mutate_random(sol: list, syntheticgene: SyntheticGene, nreplicates = 10, mutation_rate = 0.05):
  '''Take a surviving member and replicate it with the chance to mutate it'''
  replicates = []
  cod_vec = generate_codon_vec(syntheticgene.AASeq)
  for i in range(0, nreplicates):
    #Generate a random number between 0 and 1 and if it's less than mutation_rate, permute the codon seq
    new_sol = sol.copy()
    for j in range(0, len(sol)):
      if random.random() < mutation_rate:
        new_sol[j] = random.choice(cod_vec[j])
    replicates.append(new_sol)
  return replicates

def directed_evolution(sol: list, changevecs: dict, weights: dict, syntheticgene: SyntheticGene, nreplicates = 10, mutation_rate = 0.05): # Changed changevec to changevecs and added weights
  '''Intake a codon vector and its change vectors, and choose the codon to permute based on the weighed sum of change vec'''
  replicates = []
  cod_vec = generate_codon_vec(syntheticgene.AASeq)
  ranked_choices = []
  w_change = []
  # Calculate the weighted sum of change vectors for each position
  for i in range(len(sol)):
    weighted_sum = 0
    for key in changevecs.keys():
        weighted_sum += weights[key] * changevecs[key][i]
    w_change.append(weighted_sum)

  #Now rank indices based on their value in w_change
  for i in range(len(w_change)):
    ranked_choices.append((w_change[i], i))
  ranked_choices.sort(reverse=True)

  for i in range(0, nreplicates):
    #Choose an index to permute based on a random choice from the values in ranked_choice
    # We should choose based on the index, not the value
    choice_index = random.choices(range(len(ranked_choices)), weights=[rc[0] for rc in ranked_choices])[0]
    choice = ranked_choices[choice_index][1] # Get the original index

    new_sol = sol.copy()
    current_codon = sol[choice]
    codon_choices = [c for c in cod_vec[choice] if c != current_codon]
    if len(codon_choices) == 0:
      continue
    lookaheads = []
    for j in range(0, len(codon_choices)):
      lah = sol.copy()
      lah[choice] = codon_choices[j]
      lookaheads.append(lah)
    lookahead_scores = [score_changevec(lah, calculate_change_vector(lah, syntheticgene, aobjs), weights) for lah in lookaheads] # Calculate change vector and score
    #Choose the lowest score
    min_score = min(lookahead_scores)
    min_index = lookahead_scores.index(min_score)
    new_sol[choice] = codon_choices[min_index]
    replicates.append(new_sol)
  return replicates


def save_gen(pop: list, gen: int, savepath: str):
  '''Save the population for each gene'''
  assert all(type(p) == ProposedSolution for p in pop), "pop is not a list of ProposedSolution objects"
  #Dump the list of pop objects into a file named savepath_gen
  # Need to handle ProposedSolution objects, they are not directly serializable
  # Convert them to dictionaries
  pop_dict = [p.__dict__ for p in pop]
  with open(savepath + '_gen' + str(gen), 'w') as f: # Changed mode to 'w'
    json.dump(pop_dict, f)

def codvectostr(codvecs: list):
  seqs = []
  for codvec in codvecs:
    seqs.append(''.join(codvec))
  return seqs

def strtocodvec(seqs: list):
  codvecs = []
  for seq in seqs:
    codvecs.append([seq[i:i+3] for i in range(0, len(seq), 3)])
  return codvecs

def GA(syngene: SyntheticGene, seeds: list, weights, num_gens = 30):
  '''Genetic algorithm architecture'''
  #Check to see if there is a file in the synthetic gene directory named syngene.aaseq
  pop = []
  for unii in list(set(codvectostr(seeds))):
    print("unii is: ", unii)
    uni = strtocodvec([unii])[0]
    print("uni is: ", uni)
    num_duplicates = 0
    for s in seeds:
      if s == uni:
        num_duplicates += 1
    assert num_duplicates != 0
    pop.append(ProposedSolution(uni, num_duplicates, calculate_change_vector(uni, syngene, aobjs)))

  for i in range(0, num_gens):
    new_pop = []
    for p in pop:
      #Randomlly choose to use replicate and mutate or directed evolution
      if random.random() < 0.5:
        reps = replicate_and_mutate_random(p.codons, syngene)
      else:
        reps = directed_evolution(p.codons, p.changevecs, weights, syngene) # Pass weights
      for r in reps: # Iterate through reps directly
          found = False
          for j in range(0, len(pop)):
              if pop[j].codons == r:
                  pop[j].number += 1
                  found = True
                  break
          if not found:
              new_pop.append(ProposedSolution(r, 1, calculate_change_vector(r, syngene, aobjs))) # New solution has number 1

    pop.extend(new_pop) # Add new_pop to pop


    pop = kill_off(pop, weights)
    save_gen(pop, i, os.path.join("SyntheticGenes", syngene.AASeq)) # Pass savepath
  return pop



weights = {'RareCodonAnalysis': 1, 'CodonAnalysis': 1, 'CodonPairBiasAnalysis': 1, 'GCAnalysis': 1, 'KmerAnalysis': 1}
GA(syngene, seeds, weights)

#Now take the members of pop and visualize their codon constitution

#Now apply more stringent filters like spliceAI

In [ ]:
#SpliceCriteria
#Shape
#G4Boost
#MiReact
#Epitranscriptomics